In [1]:
import pickle

import pandas as pd
import numpy as np

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [2]:
from sklearn.pipeline import make_pipeline

In [3]:
from  mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri="http://127.0.0.1:5000")
experiment = client.get_experiment_by_name("green-taxi-duration")
if experiment is not None:
    client.restore_experiment(experiment.experiment_id)

In [4]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("green-taxi-duration")

2025/06/25 06:13:30 INFO mlflow.tracking.fluent: Experiment with name 'green-taxi-duration' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/oduorri/mlops-zoomcamp/mlops-zoomcamp/04-deployment/batch/mlruns/1', creation_time=1750832010720, experiment_id='1', last_update_time=1750832010720, lifecycle_stage='active', name='green-taxi-duration', tags={}>

In [5]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def prepare_dictionaries(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [6]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

dict_train = prepare_dictionaries(df_train)
dict_val = prepare_dictionaries(df_val)

In [7]:
with mlflow.start_run():
    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dict_train, y_train)
    y_pred = pipeline.predict(dict_val)

    rmse = np.sqrt(mean_squared_error(y_pred, y_val))
    print(params, rmse)
    mlflow.log_metric('rmse', rmse)

    mlflow.sklearn.log_model(pipeline, artifact_path="model")

{'max_depth': 20, 'n_estimators': 100, 'min_samples_leaf': 10, 'random_state': 0} 6.7558229919200725


2025/06/25 06:14:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run bemused-fish-282 at: http://127.0.0.1:5000/#/experiments/1/runs/3e02c6470ff243bdaae391f76c95d665
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [8]:
from mlflow.tracking import MlflowClient


In [9]:
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
RUN_ID = '3e02c6470ff243bdaae391f76c95d665'

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [14]:
path = client.download_artifacts(run_id=RUN_ID, path='model')

In [11]:
mlflow.search_runs()


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.rmse,params.min_samples_leaf,params.n_estimators,params.random_state,params.max_depth,tags.mlflow.source.name,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.source.type,tags.mlflow.log-model.history
0,3e02c6470ff243bdaae391f76c95d665,1,FINISHED,/home/oduorri/mlops-zoomcamp/mlops-zoomcamp/04...,2025-06-25 06:13:54.807000+00:00,2025-06-25 06:14:13.347000+00:00,6.755823,10,100,0,20,/home/oduorri/.local/share/virtualenvs/web-ser...,oduorri,bemused-fish-282,LOCAL,"[{""run_id"": ""3e02c6470ff243bdaae391f76c95d665""..."


In [12]:
import os

with open(os.path.join(path, 'model.pkl'), 'rb') as f_out:
    dv = pickle.load(f_out)

In [13]:
dv

Pipeline(steps=[('dictvectorizer', DictVectorizer()),
                ('randomforestregressor',
                 RandomForestRegressor(max_depth=20, min_samples_leaf=10,
                                       n_jobs=-1, random_state=0))])